<a href="https://colab.research.google.com/github/Maame-Pokuaa77/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile requirements.txt
openai
python-dotenv
pandas
matplotlib

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GroqApiKey")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


In [3]:

# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
prompt = ask_llm("What is the premier university in Ghana?")
answer = ask_llm(prompt)
print(answer)

# TODO: Print response.usage as well — how many tokens did your call consume?
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.7,
    max_tokens=500,
)
print(response.usage)

That's a great overview of the University of Ghana. To summarize, here are the key points:

1. **Founding and history**: The University of Ghana was founded in 1948 as the University College of the Gold Coast, an affiliate college of the University of London, and became a full-fledged university in 1961.
2. **Location**: The university is located in Legon, a suburb of Accra, the capital city of Ghana.
3. **Academic programs**: The university offers a wide range of undergraduate and graduate programs in various fields, including arts, social sciences, natural sciences, engineering, law, medicine, and more.
4. **Reputation**: The University of Ghana has a strong reputation for academic excellence, research, and community engagement, and is widely regarded as one of the top universities in Ghana and West Africa.
5. **Accreditation**: The university is accredited by the National Accreditation Board of Ghana.
6. **Diversity and alumni**: The university has a diverse student body and faculty

1. What is the difference between the system and user roles? Give an example of something that belongs in each.

The system role sets up the general structure for how the model is supposed to respond. It makes the model take on a specific role or persona related to a given field, using knowledge from that field to perform the task the user wants. It also defines the standard format the response should follow, along with constraints;for example, that it should stick to credible information and avoid making things up.

Example: "You are a software engineer conducting an interview. Use credible information on how junior software developer interviews are typically conducted, and don't provide any false information."

The user role,is where the actual task is specified and it tells the system exactly what to do in that particular turn.

Example: "Here is a junior software developer's CV , does it fit the job description below?"

2. What is a token, roughly? Why do API providers bill per token rather than per request?

A token is the basic unit of text ; a character, word, or subword  that results from splitting text into fragments an LLM can read, process, and generate.

API providers bill per token rather than per request because pricing based on the request alone would ignore how much work each request actually involves. A single request could contain very few tokens or a huge number of them, and if providers charged a flat rate per request, they'd run at a loss on longer ones. Billing per token lets them charge in proportion to the actual computational cost, since it's the amount of text read and generated and not the number of times a request is sent that drives the cost on their end. This makes  billing per token the more sustainable and fair model.

In [4]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
test_question = "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.

print("Temperature = 0.0")
for i in range(5):
  answer = ask_llm(test_question, temperature=0.0)
  print(f"{i+1}, {answer}")


print("Temperature = 1.2")
for i in range(5):
  answer = ask_llm(test_question, temperature=1.2)
  print(f"{i+1}, {answer}")

Temperature = 0.0
1, Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souce" is a play on the word "source," implying a reliable and trustworthy savings product.
4. **Market Mobi**: This name incorporates "mobi," short for mobile, to suggest a convenient and accessible savings product for market traders who are always on the go.
5. **Adanbo Savings**: "Adanbo" is a Ghanaian word that means "progress" or "prosperity," which could appeal to market traders looking to improve their financial situation.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word that means "honest" or "trustworthy," which could help build trust with potential customers.
7. **Traders' Fund**: 

What did you observe at each temperature?

Temperature=0.0
The outputs were highly repetitive ;3 of the 5 responses used near-identical structure and even the same top suggestions ("Makola Save"/"Makola Savings" appeared repeatedly, along with "Traders' Treasure," "Sika Saver," "MarketMate Savings"). The model consistently converged on the same most-likely answer each time, since low temperature makes it pick the highest-probability tokens rather than sampling more broadly.

Temperature=1.2
No two responses were the same. Every run produced different name suggestions, different explanations, and sometimes different structures (e.g., some included local-language terms like "SusuBox" ,"Sua" that never appeared at temperature 0). The answers were more creative and varied.

For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?



For the loan decision-support system, temperature 0.0 is the appropriate regime. If the same applicant's letter produced a different summary, different extracted data, or a different recommendation each time it was run, the system would be unreliable and untrustworthy from the loan officer's point of view.The officer needs to know that rerunning the same input gives the same output, especially since this is a decision-support tool feeding into real financial decisions. Low temperature minimizes this inconsistency and keeps outputs deterministic, factual, and reproducible, which matters far more here than creativity does.This kind of task  involves summarization, extraction and recommendation, lower hallucination risk and consistency is the priority thus, temperature = 0.0 is the right choice.

In [5]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [6]:
#Part 3.1
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002","L006"]:
  letter_text = LETTERS[letter_id]
  prompt = f"{SUMMARY_PROMPT_V1}\n\n{letter_text}"
  output = ask_llm(prompt, temperature=0.0)
  print(f"V1 output for {letter_id}")
  print(output)
  print()
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to summarize loan application letters into concise, factual briefs. "
    "Rules: be strictly factual and neutral; do not invent, assume, or infer any detail "
    "not explicitly stated in the letter; do not add opinions or recommendations; "
    "write exactly 3-4 sentences."
)

def summary_prompt_v2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    output = ask_llm(
        summary_prompt_v2(letter_text),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0,
    )
    print(f"V2 output for {letter_id}")
    print(output)
    print()
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]

    v1_out = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}", temperature=0.0)
    v2_out = ask_llm(summary_prompt_v2(letter_text), system_prompt=SUMMARY_SYSTEM_V2, temperature=0.0)

    print(f" {letter_id} ")
    print(" V1 ")
    print(v1_out)
    print("\n V2 ")
    print(v2_out)
    print("\n")


V1 output for L002
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when his finances recover.

V1 output for L006
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.

V2 output for L002
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng does not currently have collateral to offer. He expects to repay the l





1. What concrete problems did V1's output have that V2 fixed? Quote examples.

V1 added its own opinion instead of staying factual. For L006, V1 says "He has no prior experience," but the letter never actually states this, it only says he hasn't started the businesses yet. V1 turned that into a broader claim about his experience which is not backed by the letter.

For L002, V1 says "He's experiencing a slow business period," but the letter never explicitly said his business was slow, Kwame only said business has been slow in a general sense tied to needing the loan, not as a separate fact about his current period. V1 also left out that he has no collateral, which is an important detail for a loan officer, while V2 clearly stated "Mr. Boateng does not currently have collateral to offer."

V1 also varied in sentence count and structure between the two letters, while V2 consistently stayed factual, neutral, and used the right number of sentences (3 to 4) for both letters, without adding opinions or missing key details like collateral status.

2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

"No invented details" is essential because if the summarizer adds its own opinion or comes up with a detail about the applicant that is false, it can lead to wrong inference and wrong conclusions by the loan officer, which could directly affect a real financial decision made about a real person.

This failure mode is called hallucination in the LLM literature, where the system generates a plausible-sounding response that is false or not backed by any evidence, basically not a grounded fact.

In [7]:
#Part 3.2
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd

EXTRACT_SYSTEM = (
     "You are a data extraction assistant for a microfinance loan officer. "
     "You extract structured facts from loan application letters. "
     "You must respond with ONLY a valid JSON object, no explanations, no markdown fences, "
     "no extra text before or after the JSON. "
     "If a field is not explicitly stated in the letter, use null. Do not guess or infer."
)

EXTRACT_PROMPT = """Extract the following fields from the loan application letter below and return them as a JSON object with EXACTLY these keys:
- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not stated in the letter, use null. Do not guess.

Example:
"Dear Sir, My name is Ama Pokuaa, a snack vendor in Accra. I request GHS 5,000 to set up a vending space. My monthly profit is about GHS 600. My mother will guarantee the loan. I propose to repay GHS 300 monthly over 18 months."

JSON:
{{
  "applicant_name": "John Mensah",
  "amount_ghs": 5000,
  "purpose": "buy timber and tools",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}}

Now extract from this letter:

{letter_text}

JSON:"""

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)
    raw_output = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=0.0, max_tokens=500)

    #This strips markdown fences if present
    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"Warning: failed to parse JSON. Error: {e}")
        print(f"Raw output was:\n{raw_output}")
        return None
# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []
for letter_id, letter_text in LETTERS.items():
    fields = extract_fields(letter_text)
    if fields is not None:
        fields["letter_id"] = letter_id
        results.append(fields)
    else:
        results.append({"letter_id": letter_id})

df = pd.DataFrame(results)
cols = ["letter_id", "applicant_name", "amount_ghs", "purpose",
        "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]
df = df[[c for c in cols if c in df.columns]]
df

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


 1. Why must the few-shot example NOT come from the six letters you are processing?

 The few-shot example must not come from the six letters because it's like training the model to make sure it extracts the right information, so if we pass the original test cases as the example, the system might memorize it and just reproduce it, and it may not work properly on unseen data, just like the concept of overfitting.


 2. Why "use null, do not guess" — what did the model do without that instruction?
 This instruction enforces the constraint that the model should use only known facts and not give any plausible but false information, to avoid hallucination. With the instruction in place, it returned null for the parts with no value stated in the letter.


 3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?
 Temperature=0 is the right choice for extraction because it produces consistent information, it doesn't vary its outputs if the same information is passed multiple times. It's the right choice for extraction because this task has one correct answer, it returns exactly what it sees and doesn't try to be creative and come up with something out of the box.

In [8]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_SYSTEM = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to produce a decision-support brief for a loan application, based ONLY on "
    "the letter and the extracted data provided. You must be factual and grounded — do not "
    "invent details, do not speculate beyond what is stated. "
    "IMPORTANT: You do not make loan decisions. Final approval or rejection is always made "
    "by a human loan officer. You must NEVER output the words 'approve' or 'reject', and you "
    "must never recommend that the loan be granted or denied. Your job ends at surfacing "
    "information and suggesting a procedural next step."
)

BRIEF_PROMPT = """Below is a loan application letter and structured data extracted from it.
Produce a decision-support brief with EXACTLY these four sections:

1. Strengths (bullet points, grounded only in the letter)
2. Risks / Red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (choose ONE of: "invite for interview", "request additional documents",
   "flag for senior review", "request guarantor/collateral details" — do NOT say "approve" or "reject")

Letter:
{letter_text}

Extracted data (JSON):
{extracted_json}

Brief:"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

def generate_brief(letter_text, extracted_fields):
    extracted_json_str = json.dumps(extracted_fields, indent=2)
    prompt = BRIEF_PROMPT.format(letter_text=letter_text, extracted_json=extracted_json_str)
    return ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0.0, max_tokens=600)


briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)
    brief = generate_brief(letter_text, extracted)
    briefs[letter_id] = brief

for letter_id in ["L001", "L002", "L006","L003"]:
    print(f"Brief for {letter_id}")
    print(briefs[letter_id])
    print("\n")



Brief for L001
## 1. Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market.
* She has a stable business with a monthly profit of GHS 900.
* Akosua has saved GHS 2,500 with the susu scheme over two years without missing a contribution, demonstrating her ability to manage savings.
* She has a guarantor, her sister, who is a teacher, adding a level of security to the loan.

## 2. Risks / Red flags
* The loan amount of GHS 8,000 is significant compared to her monthly profit, which might pose a risk if her business does not expand as planned.
* The repayment plan of GHS 450 monthly over 20 months is approximately half of her current monthly profit, which could be challenging if her business does not grow.
* There is no detailed information on the sister's financial stability or ability to act as a guarantor.

## 3. Missing information the officer should request
* Detailed financial records of the business to assess its stability and growth 

1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each?



For L003, the strengths and red flags identified were correct. The strengths correctly captured things like the registered business status, track record, employees, sales records, and the fixed deposit collateral. The red flags, including the point about no detailed information on liabilities and cash flow, were also reasonable, since the sales records mainly show revenue and don't necessarily cover existing liabilities or a full cash flow picture.

For L006, the strengths and red flags identified were also spot on, correctly reflecting the lack of any business track record, the absence of collateral, and the unfocused scope of starting three unrelated businesses at once.



2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.

Practical reason: The model only has access to the information given in the loan application letter, and it would make its decision solely based on that, without considering other factors such as follow-up questions, interviews, or additional documents submitted later. Making a binding decision on this partial basis introduces bias, since it ignores new information a loan officer might gather by directly interacting with the applicant.

Ethical reason: If the LLM were allowed to make the decision, there would be no one clearly responsible for that action, compared to a human loan officer making the final call. There is also a discrimination risk, since the model's decision would be shaped by patterns in its training data. For example, if the training data shows loans granted to 10 males and 2 females, the model may learn to favor granting loans to males over females, reproducing that bias in its decisions.



In [9]:
prompts_content = '''"""
prompts.py - Prompt templates for the Loan Decision Support System (Lab 4)

How the prompts evolved:

Summarization:
V1 was just a one line prompt "Summarize this:", no role given, no constraints, no fixed
sentence count. Based on the output, it adds its own opinion and gives details not stated
in the letter, for example it said Kofi has no prior experience when the letter just said
he hasn't started the businesses yet. It also left out important details like Kwame having
no collateral. V2 fixed this by giving the model a role (assistant to a microfinance loan
officer) and constraints, be factual, neutral, don't invent details, and use exactly 3-4
sentences. V2 stayed factual and used the right number of sentences, so this is the version
used in the final system. Run at temperature=0.

Extraction:
Built with an explicit schema so the model knows exactly which keys to return, one few-shot
example that is NOT from the six letters (using a letter I wrote myself), because if the
example is one of the actual letters being processed, the model can just memorise it and
reproduce it instead of actually extracting, same idea as overfitting. Also added "if a
field is not stated in the letter, use null, do not guess" to stop the model from giving
plausible but false info for missing fields, which is hallucination. Run at temperature=0
because extraction has one correct answer per letter, unlike a creative task where you
want variation.

Brief:
Takes the letter and the extracted JSON together and produces 4 sections, strengths, risks,
missing information, and a suggested next step. The model is told not to output
"approve" or "reject". Practical reason: the model only sees what is in the letter, it
doesn't have access to follow up questions, interviews, or extra documents the loan officer
might get later, so letting it decide would be deciding on partial information. Ethical
reason: if the model made the decision, no one is really responsible for that decision the
way a human loan officer is responsible, and there is a risk of discrimination if the
training data has patterns like more loans historically granted to one group over another.
Run at temperature=0.
"""

SUMMARY_SYSTEM = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to summarize loan application letters into short, factual briefs. "
    "Rules: be strictly factual and neutral; do not invent, assume, or infer any detail "
    "not explicitly stated in the letter; do not add opinions or recommendations; "
    "write exactly 3-4 sentences."
)

def SUMMARY_PROMPT(letter_text):
    return f"Summarize this loan application:\\n\\n{letter_text}"


EXTRACT_SYSTEM = (
    "You are a data extraction assistant for a microfinance loan officer. "
    "You extract structured facts from loan application letters. "
    "You must respond with ONLY a valid JSON object, no explanations, no markdown fences, "
    "no extra text before or after the JSON. "
    "If a field is not explicitly stated in the letter, use null. Do not guess or infer."
)

EXTRACT_PROMPT = """Extract the following fields from the loan application letter below and return them as a JSON object with EXACTLY these keys:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not stated in the letter, use null. Do not guess.

Example:

Letter:
"Dear Sir, My name is John Mensah, a carpenter in Tema. I request GHS 5,000 to buy timber and tools. My monthly profit is about GHS 600. My brother will guarantee the loan. I propose to repay GHS 300 monthly over 18 months."

JSON:
{{
  "applicant_name": "John Mensah",
  "amount_ghs": 5000,
  "purpose": "buy timber and tools",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}}

Now extract from this letter:

{letter_text}

JSON:"""


BRIEF_SYSTEM = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to produce a decision-support brief for a loan application, based ONLY on "
    "the letter and the extracted data provided. You must be factual and grounded — do not "
    "invent details, do not speculate beyond what is stated. "
    "IMPORTANT: You do not make loan decisions. Final approval or rejection is always made "
    "by a human loan officer. You must NEVER output the words 'approve' or 'reject', and you "
    "must never recommend that the loan be granted or denied. Your job ends at surfacing "
    "information and suggesting a procedural next step."
)

BRIEF_PROMPT = """Below is a loan application letter and structured data extracted from it.
Produce a decision-support brief with EXACTLY these four sections:

1. Strengths (bullet points, grounded only in the letter)
2. Risks / Red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (choose ONE of: "invite for interview", "request additional documents",
   "flag for senior review", "request guarantor/collateral details" — do NOT say "approve" or "reject")

Letter:
{letter_text}

Extracted data (JSON):
{extracted_json}

Brief:"""
'''

with open("prompts.py", "w") as f:
    f.write(prompts_content)

print("prompts.py written.")

prompts.py written.


In [10]:
#Part 3.4 — Commit your prompt templates
#Prompts ARE code. Save your final SUMMARY_PROMPT, EXTRACT_PROMPT, and
#BRIEF_PROMPT into a separate file prompts.py (or prompts.md) in your
#repository and commit it with a message describing how the prompts evolved. Paste your commit hash below.

from google.colab import files
files.download("prompts.py")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Commit Hash : a04fe38


In [11]:
#Part 4.1
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy

#["L001", "L003", "L006"]
gold_ids = list(GOLD.keys())
fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

def values_match(field, extracted_val, gold_val):
    if field == "applicant_name":
        if extracted_val is None or gold_val is None:
            return extracted_val == gold_val
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()
    elif field == "purpose":
        #purpose is free text, so an exact match is quite strict since the model may phrase
        #things correctly but differently from how the gold label is worded. The assignment
        #doesn't ask for fuzzy matching though, so I am using case-insensitive exact match here.

        if extracted_val is None or gold_val is None:
            return extracted_val == gold_val
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()
    else:
        return extracted_val == gold_val

#Re-running extraction fresh here so I know exactly what values are being compared
#against GOLD, instead of reusing older results from earlier in the notebook.
extracted_by_id = {}
for letter_id in gold_ids:
    extracted_by_id[letter_id] = extract_fields(LETTERS[letter_id])

#comparison table consisting of rows = fields, columns = L001 / L003 / L006 / accuracy
rows = []
for field in fields:
    row = {"field": field}
    correct_count = 0
    for letter_id in gold_ids:
        extracted_val = extracted_by_id[letter_id].get(field) if extracted_by_id[letter_id] else None
        gold_val = GOLD[letter_id][field]
        match = values_match(field, extracted_val, gold_val)
        row[letter_id] = "✓" if match else f"✗ (got: {extracted_val}, gold: {gold_val})"
        if match:
            correct_count += 1
    row["accuracy"] = f"{correct_count}/3 ({correct_count/3:.0%})"
    rows.append(row)

accuracy_df = pd.DataFrame(rows)
accuracy_df = accuracy_df[["field"] + gold_ids + ["accuracy"]]
accuracy_df


,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,3/3 (100%)
1,amount_ghs,✓,✓,✓,3/3 (100%)
2,purpose,✗ (got: buy a deep freezer and expand into fro...,✗ (got: purchase two industrial sewing machine...,"✗ (got: start a car washing business, a provis...",0/3 (0%)
3,monthly_profit_ghs,✓,✓,✓,3/3 (100%)
4,has_collateral_or_guarantor,✓,✓,✓,3/3 (100%)
5,repayment_months,✓,✓,✓,3/3 (100%)


In [13]:
#Part 4.2- Reliability: is the system consistent?

# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
def run_extraction_trials(letter_text, temperature, n=5):
    # this runs the extractor n times on the same letter at a given temperature
    # so I can check if the outputs stay the same or change across runs
    results = []
    for i in range(n):
        prompt = EXTRACT_PROMPT.format(letter_text=letter_text)
        raw_output = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=temperature, max_tokens=500)

        # cleaning the output in case the model wraps the JSON in markdown fences
        cleaned = raw_output.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.strip("`")
            cleaned = cleaned.replace("json", "", 1).strip()

        try:
            parsed = json.loads(cleaned)
            results.append(parsed)
        except json.JSONDecodeError:
            # if it's not valid JSON, I store None so I can still count it as a failed run
            results.append(None)
    return results

letter_L004 = LETTERS["L004"]

# running 5 trials at temperature=0 and 5 trials at temperature=1.0 to compare consistency
results_temp0 = run_extraction_trials(letter_L004, temperature=0.0, n=5)
results_temp1 = run_extraction_trials(letter_L004, temperature=1.0, n=5)
# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

def summarize_reliability(results, label):
    # counting how many of the 5 runs actually returned valid JSON
    valid_count = sum(1 for r in results if r is not None)

    # converting each valid result into a sorted-key JSON string so I can directly
    # compare them for exact equality, since dicts with different key order
    # would otherwise look different even if the values are the same
    canonical_strings = [json.dumps(r, sort_keys=True) for r in results if r is not None]
    unique_count = len(set(canonical_strings))

    print(f" {label} ")
    print(f"Valid JSON: {valid_count}/5")
    print(f"Unique result strings among valid runs: {unique_count}")
    # if there's only 1 unique string, it means every valid run gave the exact same output
    print(f"Identical across all valid runs: {'Yes' if unique_count == 1 else 'No'}")
    print()
    for i, r in enumerate(results):
        print(f"Run {i+1}: {r}")
    print()

summarize_reliability(results_temp0, "Temperature = 0.0")
summarize_reliability(results_temp1, "Temperature = 1.0")


 Temperature = 0.0 
Valid JSON: 5/5
Unique result strings among valid runs: 1
Identical across all valid runs: Yes

Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_gh

In [14]:
#Part 4.3 — Hallucination probing

# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?

letter_L001 = LETTERS["L001"]

hallucination_test1_prompt = (
    f"Here is a loan application letter:\n\n{letter_L001}\n\n"
    "Question: What is the applicant's credit score?"
)

test1_system = (
    "You are an assistant to a microfinance loan officer. Answer questions about the "
    "loan application ONLY using information explicitly stated in the letter. "
    "If the information is not in the letter, say clearly that it is not stated. "
    "Do not guess or invent an answer."
)

test1_output = ask_llm(hallucination_test1_prompt, system_prompt=test1_system, temperature=0.0)
print(" Test 1 output ")
print(test1_output)
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
irrelevant_text = (
    "Weather report for Accra, Tuesday: Sunny with scattered clouds, high of 31°C, "
    "low of 24°C. Light winds from the southwest at 10 km/h. Humidity around 70%. "
    "No rainfall expected today. Tomorrow's forecast: partly cloudy with a slight "
    "chance of afternoon showers."
)

test2_result = extract_fields(irrelevant_text)
print("Test 2 output ")
print(test2_result)

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

 Test 1 output 
The applicant's credit score is not stated in the letter.
Test 2 output 
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


Hallucination Probing Results

Test 1: Pass
Output:The applicant's credit score is not stated in the letter.


Test 2: Pass
Output:{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}
